In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import joblib

print("Bibliothèques importées avec succès")

Bibliothèques importées avec succès


In [2]:
with open('../../backend/data/cleaned/planches_cleaned.json', 'r', encoding='utf-8') as f:
    planches = json.load(f)

print(f"Nombre de planches : {len(planches)}")

Nombre de planches : 112


In [3]:
niveau_map = {
    'Débutant - Intermédiaire': 1,
    'Intermédiaire - Confirmé': 2,
    'Confirmé - Expert': 3
}

taille_vagues_map = {
    'Petites vagues': 1,
    'Vagues moyennes': 2,
    'Grandes vagues': 3,
    'Vagues creuses': 4
}

shape_map = {
    'Longboard': 1,
    'Mini Malibu': 2,
    'Evolutive': 3,
    'Fish': 4,
    'Mid Length': 5,
    'Hybride': 6,
    'Shortboard': 7
}

rows = []
for p in planches:
    volume_moyen = np.mean(p['volume']) if p['volume'] else 0
    rows.append({
        'nom': p['nom'],
        'marque': p.get('marque', ''),
        'prix': p['prix'],
        'niveau_label': p['niveau'],
        'shape_label': p['shape'],
        'taille_vagues_label': p['taille_vagues'],
        'niveau': niveau_map.get(p['niveau'], 1),
        'shape': shape_map.get(p['shape'], 1),
        'taille_vagues': taille_vagues_map.get(p['taille_vagues'], 1),
        'volume_moyen': volume_moyen,
        'nb_derives': p['nb_derives'] if p['nb_derives'] else 3
    })

df = pd.DataFrame(rows)
print(df.shape)
print(df.head())

(112, 11)
                                              nom      marque   prix  \
0                            Odysea Log Cool Grey  Catch Surf  431.9   
1                                The Ripper Round        Lost  774.9   
2                         JJF by Pyzel- Astrofish         Jjf  479.9   
3                               Roller Royal Blue     Softech  323.9   
4  Odysea Log - Blair Conklin Pro Model Turquoise  Catch Surf  448.9   

               niveau_label  shape_label taille_vagues_label  niveau  shape  \
0  Débutant - Intermédiaire    Longboard      Petites vagues       1      1   
1  Intermédiaire - Confirmé   Shortboard      Vagues creuses       2      7   
2  Intermédiaire - Confirmé         Fish     Vagues moyennes       2      4   
3  Débutant - Intermédiaire  Mini Malibu      Petites vagues       1      2   
4  Débutant - Intermédiaire    Longboard     Vagues moyennes       1      1   

   taille_vagues  volume_moyen  nb_derives  
0              1     78.200000       

In [4]:
features = ['niveau', 'shape', 'taille_vagues', 'prix', 'volume_moyen', 'nb_derives']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# NearestNeighbors non supervisé
nn = NearestNeighbors(n_neighbors=5, metric='euclidean')
nn.fit(X_scaled)

print("Modèle entraîné avec succès")

Modèle entraîné avec succès


In [5]:
def recommander(niveau_user, budget, taille_vagues_meteo, vent_vitesse, n=5):
    
    vecteur = [[
        niveau_map.get(niveau_user, 1),
        4,
        taille_vagues_map.get(taille_vagues_meteo, 1),
        budget,
        50,
        3
    ]]
    
    vecteur_scaled = scaler.transform(vecteur)
    distances, indices = nn.kneighbors(vecteur_scaled, n_neighbors=20)
    
    candidats = df.iloc[indices[0]].copy()
    candidats['distance'] = distances[0]
    
    # Filtrer par budget
    candidats = candidats[candidats['prix'] <= budget]
    
    # Filtrer par niveau
    candidats = candidats[candidats['niveau_label'] == niveau_user]
    
    # Supprimer les doublons
    candidats = candidats.drop_duplicates(subset=['prix', 'shape_label', 'taille_vagues_label', 'volume_moyen'])
    
    # Cas 1 : budget insuffisant ou aucune planche pour ce niveau
    if candidats.empty:
        prix_min_niveau = df[df['niveau_label'] == niveau_user]['prix'].min()
        return {
            'succes': False,
            'message': f"Budget insuffisant. Nos planches {niveau_user} débutent à {prix_min_niveau}€",
            'prix_min': prix_min_niveau,
            'planches': []
        }
    
    # Cas 2 : moins de 5 planches trouvées, on retourne ce qu'on a
    resultats = candidats.head(n)
    
    return {
        'succes': True,
        'nb_resultats': len(resultats),
        'planches': resultats[['nom', 'marque', 'niveau_label', 'shape_label',
                        'taille_vagues_label', 'prix', 'volume_moyen', 'distance']].to_dict('records')
    }

In [6]:
# Sauvegarder le modèle et le scaler
joblib.dump(nn, 'modele_knn.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(df, 'planches_df.pkl')

print("✅ Modèle sauvegardé avec succès")
print("   - modele_knn.pkl")
print("   - scaler.pkl")
print("   - planches_df.pkl")

✅ Modèle sauvegardé avec succès
   - modele_knn.pkl
   - scaler.pkl
   - planches_df.pkl


In [1]:
import joblib
df_test = joblib.load('planches_df.pkl')
print(df_test.columns.tolist())
print(df_test[['nom', 'marque']].head(3))

['nom', 'marque', 'prix', 'niveau_label', 'shape_label', 'taille_vagues_label', 'niveau', 'shape', 'taille_vagues', 'volume_moyen', 'nb_derives']
                       nom      marque
0     Odysea Log Cool Grey  Catch Surf
1         The Ripper Round        Lost
2  JJF by Pyzel- Astrofish         Jjf
